## MEMO – uso della 09_03

**Una volta sola:** pre-tagging, registrazione delle 15 predeterminate (09_02). Non rilanciare.

**Parametri:** `top_j` = chunk all'LLM, `top_k` = candidati del retrieval. Si cambiano nella cella 0 (o `bench.top_j = new`).
Fissare prima del `pre_edit`; non cambiare tra `pre_edit` e `post_edit`.
Dopo averli cambiati usa un `RUN_ID` nuovo (es. `post_edit_j7`) o cancella il vecchio `*_generation.jsonl`: `generate()` salta ciò che è già salvato.

**Ordine di esecuzione**
1. Cella 0 (init), una volta per kernel.
2. Pilota (opz.): 3–4 domande con `bench.generate("pilot", query_ids=[...])`; se C0 è già ~perfetto o molto basso, rivedi `top_j`.
3. Generazione con `RUN_ID = "pre_edit"` (cella 2). Riprendibile: se si interrompe, rilancia.
4. Backup sidecar (poz.): `shutil.copy(sidecar_path, sidecar_path.with_name("dst_benchmark_pre_edit.json"))`.
5. Editing {notebook 09_03a}: cella 1a (cambia `QID`, modifica nel widget), cella 1b per controllare. `bench.edit_status()` conta solo le modifiche query↔chunk delle predq. !!!NB!!! riavviare kernel di 09_03 o non vedrà le modifiche di 09_03a.
6. Generazione con `RUN_ID = "post_edit"` (cella 2).
7. RAGAS (3.1 e 3.2) e report (4): lavorano sui file salvati, non sul sidecar. Imposta `RUN_ID` prima di ogni lancio.

**Altri esperimenti con modifiche diverse**
Cambiare `RUN_ID` rinomina ovviamente solo gli output. Necessario ripartire da una diversa istanziazione del sidecar (es la copia `dst_benchmark_pre_edit.json`); cambiare `sidecar_path` e un `RUN_ID` nuovo.

**Confronto prima/dopo**
Sulle righe C2 e C3 dei due run. C0 non dipende dalle modifiche: nel `post_edit` puoi valutare RAGAS solo su `configs=["C2_dist", "C3_dist_tags"]`.

**Attenzioni**
- `compare_retrieval_vs_reranking` (cella 1b) usa tutti i `top_k` chunk per il baseline, quindi solo per guardare, non per il confronto.
- NaN del giudice riportati e non sostituirli con 0.

In [1]:
### 0 ### Inizializzazione - eseguire solo una volta per "vita" del kernel
import sys, json, logging
from pathlib import Path

project_root = Path("~/tesi_graphrag").expanduser().resolve()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.pipeline_handler import PipelineHandler
from src.evaluation.benchmark_runner import RagasBenchmark

#"""
logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(name)s: %(message)s",
                    datefmt="%H:%M:%S", force=True)
#"""

QDIR = project_root / "data/queries/dst"
sidecar_path = project_root / "data/processed/dst/dst_benchmark.json"
queries_path = QDIR / "eval_queries.json"          # file unico (42 query): serve a update_query(new_id=...)
pred_path, norm_path = QDIR / "queries_predetermined.json", QDIR / "queries_normal.json"

handler = PipelineHandler(sidecar_path=sidecar_path, queries_path=queries_path)

bench = RagasBenchmark(
    handler, predetermined_path=pred_path, normal_path=norm_path,
    out_dir=project_root / "data/benchmark/dst",
    top_j=8, top_k=20, top_n=7,       # stesso numero di chunk (top_j) per TUTTE le configurazioni
    # gold_path=QDIR / "gold_chunks.json",   # opzionale: {query_id: [chunk_id,...]} -> metriche retrieval senza LLM
)
print("!>> Handler e benchmark pronti")

/home/jovyan/tesi_graphrag/.venv/lib/python3.12/site-packages/langchain_community/embeddings/fastembed.py:109: UserWarning: The model sentence-transformers/paraphrase-multilingual-mpnet-base-v2 now uses mean pooling instead of CLS embedding. In order to preserve the previous behaviour, consider either pinning fastembed version to 0.5.1 or using `add_custom_model` functionality.
  values["model"] = fastembed.TextEmbedding(
06:57:26 [INFO] httpx: HTTP Request: GET http://localhost:6333 "HTTP/1.1 200 OK"
06:57:26 [INFO] src.pipeline_handler: ?> Pipeline Inizializzata


!>> Handler e benchmark pronti


In [2]:
### PRE-TAGGING ### con pre-tagger specifico per questo dataset
from src.evaluation.tag_handling import pretag, explain
display(pretag(handler))                          # azzera i tag dei chunk, backup e pretagging
explain(handler, "Didimo_SeriesParallel_chunk_9") # perché un chunk ha o non ha certi tag

06:57:26 [INFO] src.pipeline_handler: ?> PreTagging Iniziato.
06:57:26 [INFO] src.pre_tagger: ?>> Inizio pre-tagging sulla collezione 'dst_for_ragas' con 7 tag
06:57:26 [INFO] httpx: HTTP Request: POST http://localhost:6333/collections/dst_for_ragas/points/count "HTTP/1.1 200 OK"
06:57:26 [INFO] httpx: HTTP Request: POST http://localhost:6333/collections/dst_for_ragas/points/scroll "HTTP/1.1 200 OK"
06:57:27 [INFO] httpx: HTTP Request: POST http://localhost:6333/collections/dst_for_ragas/points/scroll "HTTP/1.1 200 OK"
06:57:27 [INFO] httpx: HTTP Request: POST http://localhost:6333/collections/dst_for_ragas/points/scroll "HTTP/1.1 200 OK"
06:57:27 [INFO] httpx: HTTP Request: POST http://localhost:6333/collections/dst_for_ragas/points/scroll "HTTP/1.1 200 OK"
06:57:28 [INFO] httpx: HTTP Request: POST http://localhost:6333/collections/dst_for_ragas/points/scroll "HTTP/1.1 200 OK"
06:57:28 [INFO] httpx: HTTP Request: POST http://localhost:6333/collections/dst_for_ragas/points/scroll "HTTP

,tag,chunk,%
0,Orthogonal Representation & Embedding,528.00,24.9
1,Series-Parallel & Cubic Graphs,454.00,21.4
2,Rectilinear Planarity & Complexity,415.00,19.6
3,Bend Minimization & Bounds,298.00,14.1
4,Network Flow & Tamassia,123.00,5.8
5,Applications & Aesthetics,103.00,4.9
6,Topology-Shape-Metrics Pipeline,87.00,4.1
7,(chunk senza tag),834.00,39.3
8,(tag medi per chunk),0.95,NaN


06:57:33 [INFO] httpx: HTTP Request: POST http://localhost:6333/collections/dst_for_ragas/points/scroll "HTTP/1.1 200 OK"
06:57:33 [INFO] httpx: HTTP Request: POST http://localhost:6333/collections/dst_for_ragas/points/scroll "HTTP/1.1 200 OK"
06:57:33 [INFO] httpx: HTTP Request: POST http://localhost:6333/collections/dst_for_ragas/points/scroll "HTTP/1.1 200 OK"
06:57:33 [INFO] httpx: HTTP Request: POST http://localhost:6333/collections/dst_for_ragas/points/scroll "HTTP/1.1 200 OK"
06:57:33 [INFO] httpx: HTTP Request: POST http://localhost:6333/collections/dst_for_ragas/points/scroll "HTTP/1.1 200 OK"
06:57:33 [INFO] httpx: HTTP Request: POST http://localhost:6333/collections/dst_for_ragas/points/scroll "HTTP/1.1 200 OK"
06:57:33 [INFO] httpx: HTTP Request: POST http://localhost:6333/collections/dst_for_ragas/points/scroll "HTTP/1.1 200 OK"
06:57:33 [INFO] httpx: HTTP Request: POST http://localhost:6333/collections/dst_for_ragas/points/scroll "HTTP/1.1 200 OK"
06:57:33 [INFO] httpx: H

minimum number bends is one of the most studied problems in Graph Drawing. Garg and Tamassia [14] proved that this problem is NP-hard for general planar 4-graphs if the algorithm can freely choose the planar embedding; an O(n)-time algorithm exists for n-vertex planar 3-graphs [10] and an O(n3 log2  ...

*  4  Orthogonal Representation & Embedding    ['planar (embedding|representation)s?', 'plane (4-)?graphs?']
*  3  Bend Minimization & Bounds               ['(minimum|minimal|fewest|optimal) (number (of )?)?bends?', '\\bbends?\\b']
*  2  Network Flow & Tamassia                  ['tamassia']
   0  Topology-Shape-Metrics Pipeline          []
*  3  Rectilinear Planarity & Complexity       ['NP[- ](hard\\w*|complete\\w*)', '4-graphs?']
*  3  Series-Parallel & Cubic Graphs           ['series[- ]?parallel', '3-graphs?']
   0  Applications & Aesthetics                []

assegnati: ['Orthogonal Representation & Embedding', 'Bend Minimization & Bounds', 'Rectilinear Planarity & Complexity']


In [4]:
### CONFIG PREDQ ### (se non gia' fatto in 09_02) + controllo similarita' per eventuali query in più sopra threshold
#bench.register_predetermined()
print(bench.check_similarity().head(5))

06:58:51 [WARNING] src.evaluation.benchmark_runner: !> 1 query normali troppo simili a una predeterminata: ['ALG_N02']


   normal_id paired_with   max_sim closest_pred  sim_with_partner
7    ALG_N02     ALG_P02  0.812466      ALG_P02          0.812466
17   CMP_N03     CMP_P01  0.798265      DEF_P01          0.616754
6    ALG_N01     ALG_P03  0.793582      ALG_P02          0.653378
5    VIN_N02     VIN_P01  0.791388      CPX_P04          0.711623
20   CHI_N02     CHI_P01  0.748677      VIN_P02          0.424327


In [6]:
### PILOTA ### domande x (top_j = 8 e 10): serve a vedere se C0 è già al soffitto o troppo basso
import re, pandas as pd
PILOT_IDS = ["CPX_P01", "ALG_P03", "VIN_P02", "CMP_P02", "DEF_N03", "CPX_N01", "CHI_N01"] 

def coverage(ref, ctxs):
    """Proxy gratuito del context recall: quota di parole 'forti' della reference presenti nei chunk."""
    words = set(re.findall(r"[a-z0-9.]{5,}", ref.lower()))
    text = " ".join(ctxs).lower()
    return sum(w in text for w in words) / max(len(words), 1)

rows = []
for j in (8, 10):
    bench.top_j = j
    g = bench.generate(f"pilot_j{j}", query_ids=PILOT_IDS)      # run_id diverso per ogni j
    g["top_j"] = j
    g["coverage"] = [coverage(r, c) for r, c in zip(g.reference, g.retrieved_contexts)]
    rows.append(g)
pil = pd.concat(rows)

# 1) copertura media per configurazione e top_j (soffitto ~1.0 = poco margine; molto bassa = serve più j)
display(pil.pivot_table(index="config", columns="top_j", values="coverage", aggfunc="mean").round(2))
# 2) per domanda
display(pil.pivot_table(index=["query_id", "config"], columns="top_j", values="coverage").round(2))
# 3) leggi a mano 2-3 risposte baseline per capire se llama3.2 risponde in modo sensato
for _, r in pil[(pil.config == "C0_baseline") & (pil.top_j == 5)].iterrows():
    print(f"\n[{r.query_id}] {r.user_input}\n  RISPOSTA: {r.response[:400]}\n  REFERENCE: {r.reference[:300]}")
print("\nsecondi medi per risposta:", pil.groupby("top_j").seconds.mean().round(1).to_dict())

07:06:02 [INFO] src.pipeline_handler: ?> Parametri aggiornati: top_j=8, top_k=20, top_n=7, order_by=['']
07:06:02 [INFO] src.pipeline_handler: ?> Testo Query aggiornato con il testo fornito 
07:06:02 [INFO] src.pipeline_handler: ?> Vettore embedding Query aggiornato: text:Does a 4-graph always admit a rectilinear drawing?
07:06:02 [INFO] src.pipeline_handler: ?> Verifica match della nuova query con le query pre-determinate
07:06:03 [INFO] src.predetermined_query_handling_methods: <<| Match testuale esatto per Query Pre-Determinata: [predq_VIN_P02] |>>
07:06:03 [INFO] src.pipeline_handler:    >>> Query: <| Does a 4-graph always admit a rectilinear drawing? |> ricondotta a <| Does a 4-graph always admit a rectilinear drawing? |> 
07:06:03 [INFO] httpx: HTTP Request: POST http://localhost:6333/collections/dst_for_ragas/points/query "HTTP/1.1 200 OK"
07:06:03 [INFO] src.pipeline_handler: ?> Recuperati 20 record con relativi vettori di embedding
07:06:35 [INFO] httpx: HTTP Request: POST htt

top_j,8,10
config,,
C0_baseline,0.54,0.57
C1_tags,0.57,0.59
C2_dist,0.57,0.59
C3_dist_tags,0.57,0.59


top_j                    8     10
query_id config                  
ALG_P03  C0_baseline   0.71  0.76
         C1_tags       0.82  0.82
         C2_dist       0.82  0.82
         C3_dist_tags  0.82  0.82
CHI_N01  C0_baseline   0.46  0.46
         C1_tags       0.46  0.46
         C2_dist       0.46  0.46
         C3_dist_tags  0.46  0.46
CMP_P02  C0_baseline   0.29  0.29
         C1_tags       0.29  0.29
         C2_dist       0.29  0.29
         C3_dist_tags  0.29  0.29
CPX_N01  C0_baseline   0.60  0.68
         C1_tags       0.64  0.68
         C2_dist       0.64  0.68
         C3_dist_tags  0.64  0.68
CPX_P01  C0_baseline   0.60  0.65
         C1_tags       0.65  0.67
         C2_dist       0.65  0.70
         C3_dist_tags  0.65  0.67
DEF_N03  C0_baseline   0.46  0.46
         C1_tags       0.46  0.54
         C2_dist       0.46  0.54
         C3_dist_tags  0.46  0.54
VIN_P02  C0_baseline   0.67  0.67
         C1_tags       0.67  0.67
         C2_dist       0.67  0.67
         C3_dist_tags  0.67  0.67


secondi medi per risposta: {8: 28.9, 10: 35.3}


In [ ]:
### 2 ### GENERAZIONE (Ollama) - riprendibile: se si interrompe basta rilanciare la cella
RUN_ID = "pre_edit"          # poi "post_edit" dopo i ritocchi manuali
gen = bench.generate(RUN_ID)
print(gen.groupby("config").seconds.mean().round(1))
print("predq non riconosciute/errate:", gen[~gen.predq_ok].query_id.unique())
# Metriche di retrieval senza giudice (solo se gold_path impostato)
print(bench.retrieval_summary(RUN_ID))

In [ ]:
### 3.1 ### VALUTAZIONE RAGAS con giudice esterno (piu' forte di llama3.2) - una configurazione alla volta, riprendibile
# Chiavi API da variabili d'ambiente (mai nel notebook). Esempi (verifica il nome del modello attuale):
from langchain_google_genai import ChatGoogleGenerativeAI      # pip install langchain-google-genai
judge_llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0)   # GOOGLE_API_KEY
# from langchain_groq import ChatGroq; judge_llm = ChatGroq(model="llama-3.3-70b-versatile", temperature=0)

# Prova prima su 1 configurazione per stimare token e tempo; max_workers basso per i rate limit del free tier
ragas_df = bench.evaluate_ragas(RUN_ID, judge_llm, max_workers=2, configs=["C0_baseline"])

In [ ]:
### 3.2 ### Poi tutte le configurazioni
ragas_df = bench.evaluate_ragas(RUN_ID, judge_llm, max_workers=2)
print(ragas_df.isna().sum())      # NaN = fallimenti del giudice: riportali nella tesi, non sostituirli con 0

In [ ]:
### 4 ### REPORT per la tesi
ragas_df = bench.load_ragas(RUN_ID)
display(bench.summary(ragas_df, by="category"))          # per categoria (6 sezioni del professore)
display(bench.summary(ragas_df, by="is_predetermined"))  # in-sample (ritoccate) vs generalizzazione
display(bench.paired_vs_baseline(ragas_df))              # tutte le query
display(bench.paired_vs_baseline(ragas_df, subset=~ragas_df.is_predetermined))   # solo normali (generalizzazione)
ragas_df.to_csv(bench.out_dir / f"{RUN_ID}_ragas_all.csv", index=False)